<a href="https://colab.research.google.com/github/FlaviaVSC/ZIGURAT-M4T1-AULA-02/blob/main/M7T2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
 * BIMQ - Sistema de Quantificação e Orçamentação BIM
 * Lógica da Aplicação (JavaScript Vanilla)
 */
// ==========================================================================
// PARÂMETROS PADRÃO (VALORES DE FÁBRICA)
// ==========================================================================
const DEFAULT_PARAMS = {
    cost: {
        residencial: { baixo: 1750.00, medio: 2450.00, alto: 3800.00 },
        comercial: { baixo: 1980.00, medio: 2750.00, alto: 4100.00 },
        industrial: { baixo: 1450.00, medio: 1950.00, alto: 2900.00 }
    },
    concrete: {
        residencial: { baixo: 0.18, medio: 0.22, alto: 0.28 },
        comercial: { baixo: 0.20, medio: 0.24, alto: 0.30 },
        industrial: { baixo: 0.16, medio: 0.21, alto: 0.26 }
    },
    steel: {
        residencial: { baixo: 18.5, medio: 24.0, alto: 32.5 },
        comercial: { baixo: 21.0, medio: 27.5, alto: 36.0 },
        industrial: { baixo: 15.0, medio: 22.0, alto: 30.0 }
    },
    masonry: {
        residencial: { baixo: 1.2, medio: 1.5, alto: 1.8 },
        comercial: { baixo: 0.9, medio: 1.2, alto: 1.5 },
        industrial: { baixo: 0.4, medio: 0.6, alto: 0.9 }
    }
};
// ==========================================================================
// ESTADO GLOBAL DA APLICAÇÃO
// ==========================================================================
let state = {
    params: JSON.parse(localStorage.getItem('bimq_parameters')) || JSON.parse(JSON.stringify(DEFAULT_PARAMS)),
    scenarios: JSON.parse(localStorage.getItem('bimq_scenarios')) || [],
    currentCalculation: null,
    charts: {
        cost: null,
        materials: null
    }
};
// ==========================================================================
// INICIALIZAÇÃO E EVENT LISTENERS
// ==========================================================================
document.addEventListener('DOMContentLoaded', () => {
    initApp();
});
function initApp() {
    // Inicializar Tema
    initTheme();
    // Inicializar Navegação por Abas
    initTabs();
    // Inicializar Navegação Interna de Documentação
    initDocsNav();
    // Carregar Parâmetros nos Inputs
    loadParamsIntoEditor();
    // Executar Cálculo Inicial
    handleCalculation();
    // Renderizar Histórico Inicial
    renderHistory();
    // Configurar Event Listeners dos Formulários e Botões
    document.getElementById('estimator-form').addEventListener('submit', (e) => {
        e.preventDefault();
        handleCalculation();
        showToast('Cálculo atualizado com sucesso!', 'success');
    });
    document.getElementById('btn-save').addEventListener('click', saveCurrentScenario);
    document.getElementById('btn-clear-history').addEventListener('click', clearHistory);

    // Parâmetros Actions
    document.getElementById('btn-save-params').addEventListener('click', saveParameters);
    document.getElementById('btn-reset-params').addEventListener('click', resetParametersToDefault);
    // Sub-abas de Parâmetros
    const subTabButtons = document.querySelectorAll('.sub-tab-btn');
    subTabButtons.forEach(btn => {
        btn.addEventListener('click', (e) => {
            subTabButtons.forEach(b => b.classList.remove('active'));
            document.querySelectorAll('.sub-tab-content').forEach(c => c.classList.remove('active'));

            btn.classList.add('active');
            const targetId = btn.getAttribute('data-sub');
            document.getElementById(targetId).classList.add('active');
        });
    });
}
// ==========================================================================
// CONTROLE DE MODO ESCURO / CLARO
// ==========================================================================
function initTheme() {
    const themeToggle = document.getElementById('theme-toggle');
    const savedTheme = localStorage.getItem('bimq_theme') || 'dark';

    document.documentElement.setAttribute('data-theme', savedTheme);

    themeToggle.addEventListener('click', () => {
        const currentTheme = document.documentElement.getAttribute('data-theme');
        const newTheme = currentTheme === 'dark' ? 'light' : 'dark';

        document.documentElement.setAttribute('data-theme', newTheme);
        localStorage.setItem('bimq_theme', newTheme);

        // Atualiza as cores dos gráficos para combinar com o novo tema
        updateChartColors(newTheme);
        showToast(`Tema ${newTheme === 'dark' ? 'Escuro' : 'Claro'} ativado.`, 'info');
    });
}
// ==========================================================================
// SISTEMA DE NAVEGAÇÃO DE ABAS
// ==========================================================================
function initTabs() {
    const tabButtons = document.querySelectorAll('.tab-btn');
    const tabContents = document.querySelectorAll('.tab-content');
    tabButtons.forEach(btn => {
        btn.addEventListener('click', () => {
            const targetId = btn.getAttribute('data-target');

            tabButtons.forEach(b => b.classList.remove('active'));
            tabContents.forEach(content => content.classList.remove('active'));

            btn.classList.add('active');
            document.getElementById(targetId).classList.add('active');
            // Ajustar o layout do Chart.js ao mudar de aba (redesenhar)
            if (targetId === 'tab-estimator' && state.currentCalculation) {
                setTimeout(() => {
                    renderCharts(state.currentCalculation);
                }, 50);
            }
        });
    });
}
function initDocsNav() {
    const links = document.querySelectorAll('.docs-sidebar a');
    const articles = document.querySelectorAll('.doc-section');
    links.forEach(link => {
        link.addEventListener('click', (e) => {
            e.preventDefault();
            const targetId = link.getAttribute('href').substring(1);
            links.forEach(l => l.classList.remove('active'));
            articles.forEach(art => art.classList.remove('active'));
            link.classList.add('active');
            document.getElementById(targetId).classList.add('active');
            // Scroll suave dentro do contêiner de documentação
            document.getElementById(targetId).scrollIntoView({ behavior: 'smooth', block: 'start' });
        });
    });
}
// ==========================================================================
// MOTOR DE CÁLCULO
// ==========================================================================
function handleCalculation() {
    const area = parseFloat(document.getElementById('input-area').value);
    const type = document.getElementById('select-type').value;
    const standard = document.getElementById('select-standard').value;
    if (isNaN(area) || area <= 0) {
        showToast('Por favor, informe uma área válida.', 'error');
        return;
    }
    // Coleta dos Coeficientes com base no Estado Atual de Parâmetros
    const costUnit = state.params.cost[type][standard];
    const concreteUnit = state.params.concrete[type][standard];
    const steelUnit = state.params.steel[type][standard];
    const masonryUnit = state.params.masonry[type][standard];
    // Cálculos
    const costTotal = area * costUnit;
    const concreteTotal = area * concreteUnit;
    const steelTotal = area * steelUnit;
    const masonryTotal = area * masonryUnit;
    const results = {
        area,
        type,
        standard,
        costUnit,
        concreteUnit,
        steelUnit,
        masonryUnit,
        costTotal,
        concreteTotal,
        steelTotal,
        masonryTotal
    };
    state.currentCalculation = results;
    // Atualização da UI (KPIs)
    updateKPIs(results);
    // Renderização/Atualização dos Gráficos
    renderCharts(results);
}
function updateKPIs(data) {
    const formatterBRL = new Intl.NumberFormat('pt-BR', { style: 'currency', currency: 'BRL' });
    const formatterNum = new Intl.NumberFormat('pt-BR', { minimumFractionDigits: 2, maximumFractionDigits: 2 });
    document.getElementById('kpi-cost').innerText = formatterBRL.format(data.costTotal);
    document.getElementById('kpi-cost-sub').innerText = `Custo Unitário: ${formatterBRL.format(data.costUnit)}/m²`;
    document.getElementById('kpi-concrete').innerText = `${formatterNum.format(data.concreteTotal)} m³`;
    document.getElementById('kpi-concrete-sub').innerText = `Fator: ${data.concreteUnit.toFixed(3)} m³/m²`;
    document.getElementById('kpi-steel').innerText = `${formatterNum.format(data.steelTotal)} kg`;
    document.getElementById('kpi-steel-sub').innerText = `Fator: ${data.steelUnit.toFixed(1)} kg/m²`;
    document.getElementById('kpi-masonry').innerText = `${formatterNum.format(data.masonryTotal)} m²`;
    document.getElementById('kpi-masonry-sub').innerText = `Fator: ${data.masonryUnit.toFixed(2)} m²/m²`;
}
// ==========================================================================
// RENDERIZAÇÃO DE GRÁFICOS (CHART.JS)
// ==========================================================================
function renderCharts(data) {
    const isDark = document.documentElement.getAttribute('data-theme') === 'dark';
    const textColor = isDark ? '#94a3b8' : '#475569';
    const gridColor = isDark ? 'rgba(255, 255, 255, 0.05)' : 'rgba(0, 0, 0, 0.05)';
    // 1. Gráfico de Composição de Custos (Pie Chart)
    // Estimativa teórica de divisão: Estrutura (Concreto/Aço) = 32%, Alvenaria = 12%, Outros/Mão de Obra = 56%
    const structureCost = data.costTotal * 0.32;
    const masonryCost = data.costTotal * 0.12;
    const othersCost = data.costTotal * 0.56;
    const costCtx = document.getElementById('costChart').getContext('2d');
    if (state.charts.cost) {
        state.charts.cost.destroy();
    }
    state.charts.cost = new Chart(costCtx, {
        type: 'doughnut',
        data: {
            labels: ['Estrutura/Fundações', 'Alvenarias/Fechamento', 'Acabamentos/Outros'],
            datasets: [{
                data: [structureCost, masonryCost, othersCost],
                backgroundColor: [
                    '#a855f7', // Roxo
                    '#10b981', // Verde
                    '#0ea5e9'  // Azul/Ciano
                ],
                borderWidth: isDark ? 2 : 1,
                borderColor: isDark ? '#141b2d' : '#ffffff'
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: {
                    position: 'bottom',
                    labels: {
                        color: textColor,
                        font: { family: 'Plus Jakarta Sans', size: 11, weight: 600 }
                    }
                },
                tooltip: {
                    callbacks: {
                        label: function(context) {
                            let value = context.raw;
                            let percentage = ((value / data.costTotal) * 100).toFixed(1);
                            return ` ${context.label}: R$ ${value.toLocaleString('pt-BR', {minimumFractionDigits: 2, maximumFractionDigits: 2})} (${percentage}%)`;
                        }
                    }
                }
            },
            cutout: '65%'
        }
    });
    // 2. Gráfico de Materiais (Bar Chart)
    const materialsCtx = document.getElementById('materialsChart').getContext('2d');
    if (state.charts.materials) {
        state.charts.materials.destroy();
    }
    // Como as grandezas de concreto (m³), aço (kg) e alvenaria (m²) diferem muito,
    // usamos barras separadas com tooltip específico para unidade.
    state.charts.materials = new Chart(materialsCtx, {
        type: 'bar',
        data: {
            labels: ['Concreto (m³)', 'Aço (kg)', 'Alvenaria (m²)'],
            datasets: [{
                data: [data.concreteTotal, data.steelTotal, data.masonryTotal],
                backgroundColor: [
                    'rgba(168, 85, 247, 0.85)', // Roxo
                    'rgba(14, 165, 233, 0.85)', // Azul
                    'rgba(16, 185, 129, 0.85)'  // Verde
                ],
                borderRadius: 6,
                borderWidth: 0,
                barThickness: 32
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { display: false },
                tooltip: {
                    callbacks: {
                        label: function(context) {
                            let label = context.label;
                            let val = context.raw.toLocaleString('pt-BR', {maximumFractionDigits: 2});
                            if (context.dataIndex === 0) return ` Concreto: ${val} m³`;
                            if (context.dataIndex === 1) return ` Aço: ${val} kg`;
                            if (context.dataIndex === 2) return ` Alvenaria: ${val} m²`;
                            return ` ${val}`;
                        }
                    }
                }
            },
            scales: {
                x: {
                    grid: { display: false },
                    ticks: { color: textColor, font: { family: 'Plus Jakarta Sans', weight: 600 } }
                },
                y: {
                    grid: { color: gridColor },
                    ticks: { color: textColor, font: { family: 'Plus Jakarta Sans' } }
                }
            }
        }
    });
}
function updateChartColors(theme) {
    if (!state.currentCalculation) return;
    renderCharts(state.currentCalculation);
}
// ==========================================================================
// GESTÃO DE PARÂMETROS EDITÁVEIS
// ==========================================================================
function loadParamsIntoEditor() {
    const types = ['residencial', 'comercial', 'industrial'];
    const standards = ['baixo', 'medio', 'alto'];
    types.forEach(type => {
        standards.forEach(std => {
            // Cost inputs
            const costId = `p-cost-${type}-${std}`;
            const costInput = document.getElementById(costId);
            if (costInput) costInput.value = state.params.cost[type][std];
            // Concrete inputs
            const concreteId = `p-concrete-${type}-${std}`;
            const concreteInput = document.getElementById(concreteId);
            if (concreteInput) concreteInput.value = state.params.concrete[type][std];
            // Steel inputs
            const steelId = `p-steel-${type}-${std}`;
            const steelInput = document.getElementById(steelId);
            if (steelInput) steelInput.value = state.params.steel[type][std];
            // Masonry inputs
            const masonryId = `p-masonry-${type}-${std}`;
            const masonryInput = document.getElementById(masonryId);
            if (masonryInput) masonryInput.value = state.params.masonry[type][std];
        });
    });
}
function saveParameters() {
    const types = ['residencial', 'comercial', 'industrial'];
    const standards = ['baixo', 'medio', 'alto'];
    let hasError = false;
    // Criar um clone do objeto para ler os novos valores com segurança
    let updatedParams = JSON.parse(JSON.stringify(DEFAULT_PARAMS));
    types.forEach(type => {
        standards.forEach(std => {
            const costVal = parseFloat(document.getElementById(`p-cost-${type}-${std}`).value);
            const concreteVal = parseFloat(document.getElementById(`p-concrete-${type}-${std}`).value);
            const steelVal = parseFloat(document.getElementById(`p-steel-${type}-${std}`).value);
            const masonryVal = parseFloat(document.getElementById(`p-masonry-${type}-${std}`).value);
            if (isNaN(costVal) || costVal < 0 ||
                isNaN(concreteVal) || concreteVal < 0 ||
                isNaN(steelVal) || steelVal < 0 ||
                isNaN(masonryVal) || masonryVal < 0) {
                hasError = true;
            } else {
                updatedParams.cost[type][std] = costVal;
                updatedParams.concrete[type][std] = concreteVal;
                updatedParams.steel[type][std] = steelVal;
                updatedParams.masonry[type][std] = masonryVal;
            }
        });
    });
    if (hasError) {
        showToast('Erro ao salvar: verifique se existem campos vazios ou negativos.', 'error');
        return;
    }
    state.params = updatedParams;
    localStorage.setItem('bimq_parameters', JSON.stringify(state.params));

    // Recalcular no estimador principal
    handleCalculation();
    showToast('Matriz de coeficientes salva com sucesso!', 'success');
}
function resetParametersToDefault() {
    if (confirm('Deseja realmente restaurar os parâmetros para os valores padrão originais?')) {
        state.params = JSON.parse(JSON.stringify(DEFAULT_PARAMS));
        localStorage.setItem('bimq_parameters', JSON.stringify(state.params));
        loadParamsIntoEditor();
        handleCalculation();
        showToast('Parâmetros restaurados para os valores padrão de fábrica.', 'info');
    }
}
// ==========================================================================
// HISTÓRICO DE CENÁRIOS SALVOS
// ==========================================================================
function saveCurrentScenario() {
    if (!state.currentCalculation) return;
    let scenarioName = document.getElementById('input-scenario-name').value.trim();
    if (!scenarioName) {
        scenarioName = `Cenário ${state.scenarios.length + 1}`;
    }
    const timestamp = new Date().toLocaleString('pt-BR');

    const newScenario = {
        id: Date.now().toString(),
        name: scenarioName,
        data: { ...state.currentCalculation },
        timestamp
    };
    state.scenarios.unshift(newScenario); // Adiciona no início da lista
    localStorage.setItem('bimq_scenarios', JSON.stringify(state.scenarios));

    renderHistory();
    showToast(`Cenário "${scenarioName}" salvo no histórico!`, 'success');
}
function renderHistory() {
    const tbody = document.getElementById('history-tbody');
    tbody.innerHTML = '';
    if (state.scenarios.length === 0) {
        tbody.innerHTML = `
            <tr class="empty-state">
                <td colspan="8">Nenhum cenário salvo até o momento. Configure acima e clique em "Salvar Cenário".</td>
            </tr>
        `;
        return;
    }
    const formatterBRL = new Intl.NumberFormat('pt-BR', { style: 'currency', currency: 'BRL' });
    const formatterNum = new Intl.NumberFormat('pt-BR', { minimumFractionDigits: 1, maximumFractionDigits: 1 });
    state.scenarios.forEach(sc => {
        const tr = document.createElement('tr');
        tr.setAttribute('data-id', sc.id);
        // Badge classes mapping
        let typeBadgeClass = 'badge-res';
        if (sc.data.type === 'comercial') typeBadgeClass = 'badge-com';
        if (sc.data.type === 'industrial') typeBadgeClass = 'badge-ind';
        let stdBadgeClass = 'badge-med';
        if (sc.data.standard === 'baixo') stdBadgeClass = 'badge-low';
        if (sc.data.standard === 'alto') stdBadgeClass = 'badge-high';
        // Format names
        const typeFormatted = sc.data.type.charAt(0).toUpperCase() + sc.data.type.slice(1);
        const stdFormatted = sc.data.standard.charAt(0).toUpperCase() + sc.data.standard.slice(1);
        tr.innerHTML = `
            <td><strong>${sc.name}</strong></td>
            <td>${formatterNum.format(sc.data.area)} m²</td>
            <td><span class="badge ${typeBadgeClass}">${typeFormatted}</span></td>
            <td><span class="badge ${stdBadgeClass}">${stdFormatted}</span></td>
            <td><strong>${formatterBRL.format(sc.data.costTotal)}</strong></td>
            <td class="materials-summary-cell">
                Concrete: ${formatterNum.format(sc.data.concreteTotal)} m³<br>
                Steel: ${formatterNum.format(sc.data.steelTotal)} kg<br>
                Masonry: ${formatterNum.format(sc.data.masonryTotal)} m²
            </td>
            <td>${sc.timestamp}</td>
            <td>
                <div class="cell-actions">
                    <button class="btn-action load" title="Carregar no Estimador" onclick="loadScenario('${sc.id}')">
                        <i class="fa-solid fa-arrow-up-from-bracket"></i>
                    </button>
                    <button class="btn-action report" title="Exportar Relatório .TXT" onclick="exportScenarioToTxt('${sc.id}')">
                        <i class="fa-solid fa-file-arrow-down"></i>
                    </button>
                    <button class="btn-action delete" title="Excluir Cenário" onclick="deleteScenario('${sc.id}')">
                        <i class="fa-solid fa-trash"></i>
                    </button>
                </div>
            </td>
        `;
        tbody.appendChild(tr);
    });
}
window.loadScenario = function(id) {
    const sc = state.scenarios.find(s => s.id === id);
    if (!sc) return;
    document.getElementById('input-scenario-name').value = sc.name;
    document.getElementById('input-area').value = sc.data.area;
    document.getElementById('select-type').value = sc.data.type;
    document.getElementById('select-standard').value = sc.data.standard;
    handleCalculation();

    // Rolar até o formulário do estimador de forma suave
    document.querySelector('.input-pane').scrollIntoView({ behavior: 'smooth' });
    showToast(`Cenário "${sc.name}" carregado.`, 'info');
};
window.deleteScenario = function(id) {
    const sc = state.scenarios.find(s => s.id === id);
    if (!sc) return;
    if (confirm(`Deseja realmente excluir o cenário "${sc.name}" do histórico?`)) {
        state.scenarios = state.scenarios.filter(s => s.id !== id);
        localStorage.setItem('bimq_scenarios', JSON.stringify(state.scenarios));
        renderHistory();
        showToast('Cenário excluído.', 'warning');
    }
};
function clearHistory() {
    if (state.scenarios.length === 0) return;
    if (confirm('Deseja realmente limpar todo o histórico de cenários salvos? Esta operação não pode ser desfeita.')) {
        state.scenarios = [];
        localStorage.removeItem('bimq_scenarios');
        renderHistory();
        showToast('Histórico limpo.', 'warning');
    }
}
// ==========================================================================
// EXPORTAÇÃO DE RELATÓRIO EM TXT
// ==========================================================================
window.exportScenarioToTxt = function(id) {
    const sc = state.scenarios.find(s => s.id === id);
    if (!sc) return;
    const data = sc.data;
    const formatterBRL = new Intl.NumberFormat('pt-BR', { style: 'currency', currency: 'BRL' });
    const formatterNum = new Intl.NumberFormat('pt-BR', { minimumFractionDigits: 2, maximumFractionDigits: 2 });
    const reportContent = `========================================================================
             RELATÓRIO DE QUANTIFICAÇÃO E ORÇAMENTAÇÃO BIM
                          SISTEMA BIMQ
========================================================================
Cenário: ${sc.name}
Data de Gravação: ${sc.timestamp}
========================================================================
1. PARÂMETROS GERAIS DA EDIFICAÇÃO
------------------------------------------------------------------------
Área Construída total: ${formatterNum.format(data.area)} m²
Tipo de Obra: ${data.type.charAt(0).toUpperCase() + data.type.slice(1)}
Padrão de Acabamento: ${data.standard.charAt(0).toUpperCase() + data.standard.slice(1)}
2. COEFICIENTES BIM ADOTADOS
------------------------------------------------------------------------
Custo Unitário da Construção: ${formatterBRL.format(data.costUnit)} / m²
Fator de Consumo de Concreto: ${data.concreteUnit.toFixed(3)} m³ / m²
Fator de Consumo de Aço: ${data.steelUnit.toFixed(1)} kg / m²
Fator de Consumo de Alvenaria: ${data.masonryUnit.toFixed(2)} m² / m²
3. ESTIMATIVA DE QUANTITATIVOS DE MATERIAIS
------------------------------------------------------------------------
Volume Estimado de Concreto Estrutural: ${formatterNum.format(data.concreteTotal)} m³
Massa Estimada de Aço de Armadura: ${formatterNum.format(data.steelTotal)} kg
Área Estimada de Alvenarias de Vedação: ${formatterNum.format(data.masonryTotal)} m²
4. PREVISÃO FÍSICO-FINANCEIRA GLOBAL
------------------------------------------------------------------------
CUSTO TOTAL ESTIMADO DA OBRA: ${formatterBRL.format(data.costTotal)}
Custo de Estrutura e Fundações (Est. 32%): ${formatterBRL.format(data.costTotal * 0.32)}
Custo de Alvenarias e Vedação (Est. 12%): ${formatterBRL.format(data.costTotal * 0.12)}
Custo de Acabamento/Mão de Obra/Outros (Est. 56%): ${formatterBRL.format(data.costTotal * 0.56)}
------------------------------------------------------------------------
Documento gerado em formato paramétrico simplificado (LOD 100/200).
Valores preliminares sujeitos a alteração conforme avanço de projeto BIM.
Desenvolvido por: Sistema BIMQ.
========================================================================`;
    const blob = new Blob([reportContent], { type: 'text/plain;charset=utf-8' });
    const link = document.createElement('a');

    // Formatar nome do arquivo
    const fileName = `Relatorio_BIMQ_${sc.name.replace(/\s+/g, '_')}.txt`;

    link.href = URL.createObjectURL(blob);
    link.download = fileName;

    document.body.appendChild(link);
    link.click();
    document.body.removeChild(link);

    showToast(`Relatório "${fileName}" baixado com sucesso.`, 'success');
};
// ==========================================================================
// TOAST NOTIFICATIONS
// ==========================================================================
function showToast(message, type = 'info') {
    const container = document.getElementById('toast-container');
    if (!container) return;
    const toast = document.createElement('div');
    toast.className = `toast ${type}`;

    let iconClass = 'fa-circle-info';
    if (type === 'success') iconClass = 'fa-circle-check';
    if (type === 'warning') iconClass = 'fa-triangle-exclamation';
    if (type === 'error') iconClass = 'fa-circle-exclamation';
    toast.innerHTML = `
        <i class="fa-solid ${iconClass} toast-icon"></i>
        <div class="toast-message">${message}</div>
    `;
    container.appendChild(toast);
    // Remover após 4 segundos
    setTimeout(() => {
        toast.style.animation = 'fadeOut var(--transition-fast) forwards';
        setTimeout(() => {
            toast.remove();
        }, 300);
    }, 3500);
    // Remover ao clicar
    toast.addEventListener('click', () => {
        toast.remove();
    });
}
<!DOCTYPE html>
<html lang="pt-BR" data-theme="dark">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>BIMQ - Sistema de Quantificação e Orçamentação BIM</title>
    <!-- Google Fonts -->
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700;800&family=Plus+Jakarta+Sans:wght@300;400;500;600;700&display=swap" rel="stylesheet">
    <!-- Font Awesome Icons -->
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <!-- Chart.js CDN -->
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link rel="stylesheet" href="styles.css">
</head>
<body>
    <!-- Top Gradient Accent -->
    <div class="top-accent"></div>
    <!-- Header Navigation -->
    <header class="main-header">
        <div class="logo-area">
            <div class="logo-icon">
                <i class="fa-solid fa-cubes-stacked"></i>
            </div>
            <div class="logo-text">
                <h1>BIM<span>Q</span></h1>
                <p>Quantificação & Orçamentação</p>
            </div>
        </div>

        <nav class="nav-tabs">
            <button class="tab-btn active" data-target="tab-estimator">
                <i class="fa-solid fa-calculator"></i> Estimador
            </button>
            <button class="tab-btn" data-target="tab-parameters">
                <i class="fa-solid fa-sliders"></i> Parâmetros
            </button>
            <button class="tab-btn" data-target="tab-docs">
                <i class="fa-solid fa-book-open"></i> Documentação & One Page
            </button>
        </nav>
        <div class="header-actions">
            <button id="theme-toggle" class="action-btn-circle" title="Alternar Tema">
                <i class="fa-solid fa-sun light-icon"></i>
                <i class="fa-solid fa-moon dark-icon"></i>
            </button>
        </div>
    </header>
    <!-- Main Content Container -->
    <main class="main-container">

        <!-- ================= TAB 1: ESTIMATOR ================= -->
        <section id="tab-estimator" class="tab-content active">
            <div class="estimator-layout">
                <!-- Left Pane: Inputs -->
                <div class="input-pane glass-panel">
                    <div class="panel-header">
                        <i class="fa-solid fa-compass-drafting header-icon"></i>
                        <h2>Configuração do Projeto</h2>
                    </div>

                    <form id="estimator-form" class="form-container">
                        <div class="form-group">
                            <label for="input-scenario-name">Nome do Cenário</label>
                            <div class="input-wrapper">
                                <i class="fa-solid fa-tag input-icon"></i>
                                <input type="text" id="input-scenario-name" placeholder="Ex: Casa Unifamiliar Mod 1" value="Cenário Inicial">
                            </div>
                        </div>
                        <div class="form-group">
                            <label for="input-area">Área Construída (m²)</label>
                            <div class="input-wrapper">
                                <i class="fa-solid fa-ruler-combined input-icon"></i>
                                <input type="number" id="input-area" min="10" max="1000000" step="0.01" required placeholder="Digite a área em m²" value="150">
                            </div>
                            <span class="input-help">Área total de projeção dos pisos</span>
                        </div>
                        <div class="form-row">
                            <div class="form-group">
                                <label for="select-type">Tipo de Obra</label>
                                <div class="input-wrapper">
                                    <i class="fa-solid fa-building input-icon"></i>
                                    <select id="select-type">
                                        <option value="residencial" selected>Residencial</option>
                                        <option value="comercial">Comercial</option>
                                        <option value="industrial">Industrial</option>
                                    </select>
                                </div>
                            </div>
                            <div class="form-group">
                                <label for="select-standard">Padrão de Acabamento</label>
                                <div class="input-wrapper">
                                    <i class="fa-solid fa-award input-icon"></i>
                                    <select id="select-standard">
                                        <option value="baixo">Baixo</option>
                                        <option value="medio" selected>Médio</option>
                                        <option value="alto">Alto</option>
                                    </select>
                                </div>
                            </div>
                        </div>
                        <div class="form-actions">
                            <button type="submit" id="btn-calculate" class="btn btn-primary">
                                <i class="fa-solid fa-gears"></i> Calcular
                            </button>
                            <button type="button" id="btn-save" class="btn btn-secondary">
                                <i class="fa-solid fa-floppy-disk"></i> Salvar Cenário
                            </button>
                        </div>
                    </form>
                </div>
                <!-- Right Pane: Results -->
                <div class="results-pane">
                    <!-- KPIs Grid -->
                    <div class="kpi-grid">
                        <div class="kpi-card glass-panel highlight-cost">
                            <div class="kpi-header">
                                <span>Custo Estimado</span>
                                <div class="kpi-icon"><i class="fa-solid fa-dollar-sign"></i></div>
                            </div>
                            <div class="kpi-value" id="kpi-cost">R$ 0,00</div>
                            <div class="kpi-subtext" id="kpi-cost-sub">Custo Unitário: R$ 0,00/m²</div>
                        </div>
                        <div class="kpi-card glass-panel">
                            <div class="kpi-header">
                                <span>Concreto</span>
                                <div class="kpi-icon"><i class="fa-solid fa-trowel-bricks"></i></div>
                            </div>
                            <div class="kpi-value" id="kpi-concrete">0,00 m³</div>
                            <div class="kpi-subtext" id="kpi-concrete-sub">Fator: 0,00 m³/m²</div>
                        </div>
                        <div class="kpi-card glass-panel">
                            <div class="kpi-header">
                                <span>Aço</span>
                                <div class="kpi-icon"><i class="fa-solid fa-cubes"></i></div>
                            </div>
                            <div class="kpi-value" id="kpi-steel">0,00 kg</div>
                            <div class="kpi-subtext" id="kpi-steel-sub">Fator: 0,00 kg/m²</div>
                        </div>
                        <div class="kpi-card glass-panel">
                            <div class="kpi-header">
                                <span>Alvenaria</span>
                                <div class="kpi-icon"><i class="fa-solid fa-border-all"></i></div>
                            </div>
                            <div class="kpi-value" id="kpi-masonry">0,00 m²</div>
                            <div class="kpi-subtext" id="kpi-masonry-sub">Fator: 0,00 m²/m²</div>
                        </div>
                    </div>
                    <!-- Charts Grid -->
                    <div class="charts-grid">
                        <div class="chart-container glass-panel">
                            <h3><i class="fa-solid fa-chart-pie"></i> Composição Estimada de Custos</h3>
                            <div class="chart-wrapper">
                                <canvas id="costChart"></canvas>
                            </div>
                        </div>
                        <div class="chart-container glass-panel">
                            <h3><i class="fa-solid fa-chart-simple"></i> Resumo de Quantitativos</h3>
                            <div class="chart-wrapper">
                                <canvas id="materialsChart"></canvas>
                            </div>
                        </div>
                    </div>
                </div>
            </div>
            <!-- Bottom: Saved Scenarios History -->
            <div class="history-section glass-panel">
                <div class="section-header">
                    <div class="header-left">
                        <i class="fa-solid fa-clock-rotate-left"></i>
                        <h2>Histórico de Cenários Salvos</h2>
                    </div>
                    <button id="btn-clear-history" class="btn btn-danger-text btn-sm">
                        <i class="fa-solid fa-trash-can"></i> Limpar Tudo
                    </button>
                </div>
                <div class="table-container">
                    <table class="history-table" id="scenarios-table">
                        <thead>
                            <tr>
                                <th>Nome do Cenário</th>
                                <th>Área (m²)</th>
                                <th>Tipo</th>
                                <th>Padrão</th>
                                <th>Custo Total</th>
                                <th>Materiais Estimados</th>
                                <th>Data/Hora</th>
                                <th>Ações</th>
                            </tr>
                        </thead>
                        <tbody id="history-tbody">
                            <!-- Dynamically populated -->
                            <tr class="empty-state">
                                <td colspan="8">Nenhum cenário salvo até o momento. Configure acima e clique em "Salvar Cenário".</td>
                            </tr>
                        </tbody>
                    </table>
                </div>
            </div>
        </section>
        <!-- ================= TAB 2: PARAMETERS ================= -->
        <section id="tab-parameters" class="tab-content">
            <div class="parameters-container glass-panel">
                <div class="section-header">
                    <div>
                        <h2><i class="fa-solid fa-sliders"></i> Matriz de Coeficientes e Parametrização</h2>
                        <p class="subtitle">Ajuste os consumos médios e valores por m² de acordo com as especificações locais de seu projeto BIM.</p>
                    </div>
                    <div class="header-actions">
                        <button id="btn-reset-params" class="btn btn-secondary">
                            <i class="fa-solid fa-rotate-left"></i> Padrões de Fábrica
                        </button>
                        <button id="btn-save-params" class="btn btn-primary">
                            <i class="fa-solid fa-check"></i> Salvar Alterações
                        </button>
                    </div>
                </div>
                <div class="tabs-subnav">
                    <button class="sub-tab-btn active" data-sub="sub-cost">Custo Unitário (R$/m²)</button>
                    <button class="sub-tab-btn" data-sub="sub-concrete">Concreto (m³/m²)</button>
                    <button class="sub-tab-btn" data-sub="sub-steel">Aço (kg/m²)</button>
                    <button class="sub-tab-btn" data-sub="sub-masonry">Alvenaria (m²/m²)</button>
                </div>
                <!-- Parameters Editor Panel -->
                <div class="params-editor-body">
                    <!-- Cost Parameter Table -->
                    <div id="sub-cost" class="sub-tab-content active">
                        <div class="table-info">
                            <i class="fa-solid fa-info-circle"></i>
                            <span>Define o custo básico de construção por metro quadrado (equivalente ao CUB/m² ajustado por tipo de obra e padrão).</span>
                        </div>
                        <table class="params-table">
                            <thead>
                                <tr>
                                    <th>Tipo de Obra</th>
                                    <th>Padrão Baixo (R$)</th>
                                    <th>Padrão Médio (R$)</th>
                                    <th>Padrão Alto (R$)</th>
                                </tr>
                            </thead>
                            <tbody>
                                <tr>
                                    <td><strong>Residencial</strong></td>
                                    <td><input type="number" step="0.01" id="p-cost-residencial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-cost-residencial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-cost-residencial-alto" class="param-input"></td>
                                </tr>
                                <tr>
                                    <td><strong>Comercial</strong></td>
                                    <td><input type="number" step="0.01" id="p-cost-comercial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-cost-comercial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-cost-comercial-alto" class="param-input"></td>
                                </tr>
                                <tr>
                                    <td><strong>Industrial</strong></td>
                                    <td><input type="number" step="0.01" id="p-cost-industrial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-cost-industrial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-cost-industrial-alto" class="param-input"></td>
                                </tr>
                            </tbody>
                        </table>
                    </div>
                    <!-- Concrete Parameter Table -->
                    <div id="sub-concrete" class="sub-tab-content">
                        <div class="table-info">
                            <i class="fa-solid fa-info-circle"></i>
                            <span>Define o consumo volumétrico estimado de concreto estrutural por metro quadrado de área construída.</span>
                        </div>
                        <table class="params-table">
                            <thead>
                                <tr>
                                    <th>Tipo de Obra</th>
                                    <th>Padrão Baixo (m³/m²)</th>
                                    <th>Padrão Médio (m³/m²)</th>
                                    <th>Padrão Alto (m³/m²)</th>
                                </tr>
                            </thead>
                            <tbody>
                                <tr>
                                    <td><strong>Residencial</strong></td>
                                    <td><input type="number" step="0.001" id="p-concrete-residencial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.001" id="p-concrete-residencial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.001" id="p-concrete-residencial-alto" class="param-input"></td>
                                </tr>
                                <tr>
                                    <td><strong>Comercial</strong></td>
                                    <td><input type="number" step="0.001" id="p-concrete-comercial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.001" id="p-concrete-comercial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.001" id="p-concrete-comercial-alto" class="param-input"></td>
                                </tr>
                                <tr>
                                    <td><strong>Industrial</strong></td>
                                    <td><input type="number" step="0.001" id="p-concrete-industrial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.001" id="p-concrete-industrial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.001" id="p-concrete-industrial-alto" class="param-input"></td>
                                </tr>
                            </tbody>
                        </table>
                    </div>
                    <!-- Steel Parameter Table -->
                    <div id="sub-steel" class="sub-tab-content">
                        <div class="table-info">
                            <i class="fa-solid fa-info-circle"></i>
                            <span>Define a quantidade de quilos (kg) de armadura de aço estrutural demandada por metro quadrado de área construída.</span>
                        </div>
                        <table class="params-table">
                            <thead>
                                <tr>
                                    <th>Tipo de Obra</th>
                                    <th>Padrão Baixo (kg/m²)</th>
                                    <th>Padrão Médio (kg/m²)</th>
                                    <th>Padrão Alto (kg/m²)</th>
                                </tr>
                            </thead>
                            <tbody>
                                <tr>
                                    <td><strong>Residencial</strong></td>
                                    <td><input type="number" step="0.01" id="p-steel-residencial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-steel-residencial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-steel-residencial-alto" class="param-input"></td>
                                </tr>
                                <tr>
                                    <td><strong>Comercial</strong></td>
                                    <td><input type="number" step="0.01" id="p-steel-comercial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-steel-comercial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-steel-comercial-alto" class="param-input"></td>
                                </tr>
                                <tr>
                                    <td><strong>Industrial</strong></td>
                                    <td><input type="number" step="0.01" id="p-steel-industrial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-steel-industrial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-steel-industrial-alto" class="param-input"></td>
                                </tr>
                            </tbody>
                        </table>
                    </div>
                    <!-- Masonry Parameter Table -->
                    <div id="sub-masonry" class="sub-tab-content">
                        <div class="table-info">
                            <i class="fa-solid fa-info-circle"></i>
                            <span>Define a metragem quadrada total de alvenaria de vedação/estrutural necessária por metro quadrado de área construída.</span>
                        </div>
                        <table class="params-table">
                            <thead>
                                <tr>
                                    <th>Tipo de Obra</th>
                                    <th>Padrão Baixo (m²/m²)</th>
                                    <th>Padrão Médio (m²/m²)</th>
                                    <th>Padrão Alto (m²/m²)</th>
                                </tr>
                            </thead>
                            <tbody>
                                <tr>
                                    <td><strong>Residencial</strong></td>
                                    <td><input type="number" step="0.01" id="p-masonry-residencial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-masonry-residencial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-masonry-residencial-alto" class="param-input"></td>
                                </tr>
                                <tr>
                                    <td><strong>Comercial</strong></td>
                                    <td><input type="number" step="0.01" id="p-masonry-comercial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-masonry-comercial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-masonry-comercial-alto" class="param-input"></td>
                                </tr>
                                <tr>
                                    <td><strong>Industrial</strong></td>
                                    <td><input type="number" step="0.01" id="p-masonry-industrial-baixo" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-masonry-industrial-medio" class="param-input"></td>
                                    <td><input type="number" step="0.01" id="p-masonry-industrial-alto" class="param-input"></td>
                                </tr>
                            </tbody>
                        </table>
                    </div>
                </div>
            </div>
        </section>
        <!-- ================= TAB 3: DOCUMENTATION & ONE PAGE ================= -->
        <section id="tab-docs" class="tab-content">
            <div class="docs-layout">
                <!-- Navigation panel for Docs -->
                <aside class="docs-sidebar glass-panel">
                    <h3>Índice</h3>
                    <ul>
                        <li><a href="#doc-onepage" class="active"><i class="fa-solid fa-file-invoice"></i> Resumo One Page</a></li>
                        <li><a href="#doc-intro"><i class="fa-solid fa-circle-info"></i> Introdução e Objetivo</a></li>
                        <li><a href="#doc-architecture"><i class="fa-solid fa-layer-group"></i> Arquitetura e Stack</a></li>
                        <li><a href="#doc-calculations"><i class="fa-solid fa-calculator"></i> Metodologia de Cálculo</a></li>
                        <li><a href="#doc-future"><i class="fa-solid fa-rocket"></i> Próximas Evoluções</a></li>
                    </ul>
                </aside>
                <!-- Document Content Panel -->
                <div class="docs-content glass-panel">

                    <!-- One Page Summary -->
                    <article id="doc-onepage" class="doc-section active">
                        <div class="doc-badge">DOCUMENTO OFICIAL</div>
                        <h2>Resumo Executivo One Page</h2>
                        <p class="section-lead">Registro do processo de trabalho, diagnóstico do problema e solução proposta pelo projeto.</p>

                        <div class="one-page-grid">
                            <div class="op-card">
                                <h4><i class="fa-solid fa-triangle-exclamation text-warning"></i> O Problema</h4>
                                <p>Na fase preliminar de empreendimentos imobiliários, engenheiros, arquitetos e orçamentistas carecem de ferramentas ágeis que cruzem conceitos BIM e taxas de produtividade física para estimar de maneira realista os consumos de materiais. As planilhas tradicionais costumam ser estáticas, complexas, não centralizadas e de difícil visualização analítica (dashboards).</p>
                            </div>
                            <div class="op-card">
                                <h4><i class="fa-solid fa-lightbulb text-success"></i> A Solução</h4>
                                <p>Uma aplicação web Single Page responsiva baseada em modelagem de dados paramétrica de construção civil. Ela traduz dimensões básicas da obra (área, tipo e padrão de acabamento) em volumes brutos de insumos essenciais (concreto, aço, alvenaria) e custo monetário final. Fornece ainda comparação dinâmica gráfica e gravação local de cenários de engenharia.</p>
                            </div>
                            <div class="op-card">
                                <h4><i class="fa-solid fa-spinner text-primary"></i> Processo de Trabalho</h4>
                                <p>O desenvolvimento foi realizado de forma iterativa:</p>
                                <ol>
                                    <li><strong>Análise & Fórmulas:</strong> Mapeamento das médias de consumo físico de materiais com base no histórico da construção nacional (referências de CUB e SINAPI).</li>
                                    <li><strong>Design System:</strong> Definição de layout com painel de vidro (glassmorphism), modo escuro focado em produtividade industrial e contraste adequado.</li>
                                    <li><strong>Desenvolvimento:</strong> Codificação do motor de cálculo e persistência assíncrona.</li>
                                    <li><strong>Visualização:</strong> Implementação de gráficos com Chart.js.</li>
                                </ol>
                            </div>
                            <div class="op-card">
                                <h4><i class="fa-solid fa-square-poll-vertical text-info"></i> Resultados Obtidos</h4>
                                <ul>
                                    <li>Estimação ágil de concreto, aço e alvenaria em menos de 1 segundo.</li>
                                    <li>Previsão financeira de custo global parametrizável de acordo com o CUB da região.</li>
                                    <li>Ambiente para validação rápida de hipóteses ("E se aumentarmos a área?") com exportação direta para arquivos de texto (.txt) que servem de memoriais.</li>
                                </ul>
                            </div>
                        </div>
                    </article>
                    <!-- Technical Intro -->
                    <article id="doc-intro" class="doc-section">
                        <h2>Introdução e Objetivo</h2>
                        <p>O <strong>Sistema BIM de Quantificação e Orçamentação Paramétrica</strong> foi projetado para atuar como ferramenta de apoio à tomada de decisão conceitual (fases iniciais de projeto - LOD 100/200).</p>
                        <h3>Fluxo de Funcionamento</h3>
                        <div class="flow-chart-container">
                            <div class="flow-step">
                                <div class="step-num">1</div>
                                <h5>Entradas</h5>
                                <p>Área (m²), Tipo de Obra, Padrão de Acabamento</p>
                            </div>
                            <div class="flow-arrow"><i class="fa-solid fa-chevron-right"></i></div>
                            <div class="flow-step">
                                <div class="step-num">2</div>
                                <h5>Processamento</h5>
                                <p>Fórmula Paramétrica BIM + Matriz de Coeficientes</p>
                            </div>
                            <div class="flow-arrow"><i class="fa-solid fa-chevron-right"></i></div>
                            <div class="flow-step">
                                <div class="step-num">3</div>
                                <h5>Saídas</h5>
                                <p>Quantitativos de Concreto, Aço, Alvenaria e Custo Total</p>
                            </div>
                        </div>
                    </article>
                    <!-- Architecture and Stack -->
                    <article id="doc-architecture" class="doc-section">
                        <h2>Arquitetura e Stack do Projeto</h2>
                        <p>A stack tecnológica selecionada prioriza a simplicidade de implantação, leveza de execução física e alta flexibilidade estética:</p>

                        <table class="stack-table">
                            <thead>
                                <tr>
                                    <th>Módulo</th>
                                    <th>Tecnologia Adotada</th>
                                    <th>Propósito</th>
                                </tr>
                            </thead>
                            <tbody>
                                <tr>
                                    <td><strong>Estrutura</strong></td>
                                    <td>HTML5 Semântico</td>
                                    <td>Fornecer o esqueleto estrutural da página e organização de abas.</td>
                                </tr>
                                <tr>
                                    <td><strong>Estilização</strong></td>
                                    <td>CSS3 Vanilla (Variáveis, Flexbox, CSS Grid)</td>
                                    <td>Garante design responsivo avançado, transições fluidas de elementos e suporte a múltiplos temas nativos.</td>
                                </tr>
                                <tr>
                                    <td><strong>Comportamento</strong></td>
                                    <td>JavaScript Moderno (ES6+)</td>
                                    <td>Processa os cálculos matemáticos instantaneamente, gerencia os dados do LocalStorage e atualiza o DOM sem refresh.</td>
                                </tr>
                                <tr>
                                    <td><strong>Gráficos</strong></td>
                                    <td>Chart.js (HTML5 Canvas)</td>
                                    <td>Gera gráficos de composição orçamentária e comparativos volumétricos.</td>
                                </tr>
                                <tr>
                                    <td><strong>Iconografia</strong></td>
                                    <td>Font Awesome 6</td>
                                    <td>Oferece ícones intuitivos para enriquecimento visual do painel.</td>
                                </tr>
                            </tbody>
                        </table>
                    </article>
                    <!-- Methodology of Calculations -->
                    <article id="doc-calculations" class="doc-section">
                        <h2>Metodologia de Cálculo</h2>
                        <p>Os cálculos são realizados com base em coeficientes estatísticos consolidados por engenheiros civis na implantação de orçamentos expeditos:</p>

                        <div class="formula-box">
                            <h4>Equação de Cubagem Financeira</h4>
                            <code>Custo Total (R$) = Área (m²) × Custo Unitário Básico (R$/m²)</code>
                        </div>
                        <div class="formula-box">
                            <h4>Equações de Consumo Físico de Insumos</h4>
                            <ul>
                                <li><code>Concreto Estrutural (m³) = Área (m²) × Coeficiente Concreto (m³/m²)</code></li>
                                <li><code>Aço de Armadura (kg) = Área (m²) × Coeficiente Aço (kg/m²)</code></li>
                                <li><code>Alvenaria de Vedação (m²) = Área (m²) × Coeficiente Alvenaria (m²/m²)</code></li>
                            </ul>
                        </div>
                        <p><strong>Nota técnica:</strong> Por padrão, estes coeficientes variam incrementalmente de acordo com a sofisticação estrutural e arquitetônica (Padrões Baixo, Médio e Alto), mas o engenheiro tem total autonomia na aba <strong>Parâmetros</strong> para editar as taxas de consumo de acordo com as especificações do sistema construtivo real (ex: Alvenaria Estrutural vs. Concreto Armado tradicional).</p>
                    </article>
                    <!-- Future Evolution -->
                    <article id="doc-future" class="doc-section">
                        <h2>Próximas Evoluções (Roadmap)</h2>
                        <div class="roadmap-list">
                            <div class="roadmap-item">
                                <span class="badge badge-road">Fase 2</span>
                                <h4>Integração Direta com Arquivos IFC (BIM)</h4>
                                <p>Leitura de arquivos OpenBIM (.ifc) no navegador utilizando a biblioteca <code>web-ifc</code> para extração automática da área de projeção e volume de elementos de forma 100% digital.</p>
                            </div>
                            <div class="roadmap-item">
                                <span class="badge badge-road">Fase 3</span>
                                <h4>Banco de Dados Distribuído e Compartilhamento</h4>
                                <p>Substituir a gravação baseada em navegador (localStorage) por uma API em Node.js com banco de dados PostgreSQL, permitindo cooperação entre equipes e link de compartilhamento público de estimativas.</p>
                            </div>
                            <div class="roadmap-item">
                                <span class="badge badge-road">Fase 4</span>
                                <h4>Conexão com APIs Oficiais de Preços</h4>
                                <p>Sincronização em tempo real com bases SINAPI (Caixa Econômica Federal) e CUB regional de cada sindicato da construção civil por meio de web scraping ou APIs parceiras.</p>
                            </div>
                        </div>
                    </article>
                </div>
            </div>
        </section>
    </main>
    <!-- Toast Notification (Popup flutuante de sucesso/erro) -->
    <div id="toast-container" class="toast-container"></div>
    <script src="app.js"></script>
</body>
</html>
/* ==========================================================================
   DESIGN SYSTEM & VARIABLES
   ========================================================================== */
:root {
    /* Fonts */
    --font-primary: 'Plus Jakarta Sans', sans-serif;
    --font-secondary: 'Outfit', sans-serif;
    /* Theme: Dark (Default) */
    --bg-base: #0b0f19;
    --bg-surface: rgba(20, 27, 45, 0.7);
    --bg-surface-solid: #141b2d;
    --border-color: rgba(255, 255, 255, 0.08);
    --border-color-hover: rgba(255, 255, 255, 0.15);

    --text-primary: #f8fafc;
    --text-secondary: #94a3b8;
    --text-muted: #64748b;

    --primary: #0ea5e9;
    --primary-hover: #38bdf8;
    --primary-gradient: linear-gradient(135deg, #0ea5e9, #6366f1);
    --primary-gradient-hover: linear-gradient(135deg, #38bdf8, #818cf8);

    --accent-purple: #a855f7;
    --accent-emerald: #10b981;
    --accent-amber: #f59e0b;
    --accent-rose: #f43f5e;

    --shadow-sm: 0 2px 8px -2px rgba(0, 0, 0, 0.5);
    --shadow-md: 0 12px 24px -4px rgba(0, 0, 0, 0.4);
    --shadow-lg: 0 20px 40px -8px rgba(0, 0, 0, 0.6);
    --shadow-glow: 0 0 20px rgba(14, 165, 233, 0.15);

    --glass-blur: blur(16px);
    --transition-fast: 0.2s ease;
    --transition-normal: 0.3s cubic-bezier(0.4, 0, 0.2, 1);
    --border-radius-sm: 8px;
    --border-radius-md: 16px;
    --border-radius-lg: 24px;
}
/* Light Theme Variables */
[data-theme="light"] {
    --bg-base: #f3f4f6;
    --bg-surface: rgba(255, 255, 255, 0.75);
    --bg-surface-solid: #ffffff;
    --border-color: rgba(0, 0, 0, 0.08);
    --border-color-hover: rgba(0, 0, 0, 0.15);

    --text-primary: #0f172a;
    --text-secondary: #475569;
    --text-muted: #94a3b8;

    --primary: #0284c7;
    --primary-hover: #0ea5e9;
    --primary-gradient: linear-gradient(135deg, #0284c7, #4f46e5);
    --primary-gradient-hover: linear-gradient(135deg, #0ea5e9, #6366f1);

    --accent-purple: #8b5cf6;
    --accent-emerald: #059669;
    --accent-amber: #d97706;
    --accent-rose: #e11d48;

    --shadow-sm: 0 2px 8px -2px rgba(0, 0, 0, 0.05);
    --shadow-md: 0 12px 24px -4px rgba(0, 0, 0, 0.08);
    --shadow-lg: 0 20px 40px -8px rgba(0, 0, 0, 0.12);
    --shadow-glow: 0 0 20px rgba(2, 132, 199, 0.15);
}
/* ==========================================================================
   RESET & BASE STYLES
   ========================================================================== */
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}
html {
    scroll-behavior: smooth;
}
body {
    background-color: var(--bg-base);
    color: var(--text-primary);
    font-family: var(--font-primary);
    min-height: 100vh;
    display: flex;
    flex-direction: column;
    overflow-x: hidden;
    transition: background-color var(--transition-normal), color var(--transition-normal);
}
/* Custom Scrollbar */
::-webkit-scrollbar {
    width: 8px;
    height: 8px;
}
::-webkit-scrollbar-track {
    background: var(--bg-base);
}
::-webkit-scrollbar-thumb {
    background: var(--border-color-hover);
    border-radius: var(--border-radius-sm);
}
::-webkit-scrollbar-thumb:hover {
    background: var(--text-muted);
}
/* Top Glow Effect */
.top-accent {
    position: absolute;
    top: 0;
    left: 10%;
    width: 80%;
    height: 4px;
    background: var(--primary-gradient);
    filter: blur(1px);
    z-index: 10;
}
/* ==========================================================================
   LAYOUT: MAIN HEADER
   ========================================================================== */
.main-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 1.25rem 2.5rem;
    background: var(--bg-surface);
    backdrop-filter: var(--glass-blur);
    -webkit-backdrop-filter: var(--glass-blur);
    border-bottom: 1px solid var(--border-color);
    position: sticky;
    top: 0;
    z-index: 100;
    box-shadow: var(--shadow-sm);
    transition: background var(--transition-normal), border var(--transition-normal);
}
.logo-area {
    display: flex;
    align-items: center;
    gap: 0.75rem;
}
.logo-icon {
    font-size: 1.75rem;
    background: var(--primary-gradient);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    display: flex;
    align-items: center;
    justify-content: center;
    filter: drop-shadow(0 2px 8px rgba(14, 165, 233, 0.3));
}
.logo-text h1 {
    font-family: var(--font-secondary);
    font-size: 1.5rem;
    font-weight: 800;
    line-height: 1.1;
    letter-spacing: -0.5px;
}
.logo-text h1 span {
    color: var(--primary);
}
.logo-text p {
    font-size: 0.7rem;
    text-transform: uppercase;
    letter-spacing: 1.5px;
    color: var(--text-secondary);
    font-weight: 600;
}
/* Tab Navigation buttons */
.nav-tabs {
    display: flex;
    background: rgba(0, 0, 0, 0.15);
    padding: 0.35rem;
    border-radius: var(--border-radius-md);
    border: 1px solid var(--border-color);
    gap: 0.25rem;
}
[data-theme="light"] .nav-tabs {
    background: rgba(0, 0, 0, 0.05);
}
.tab-btn {
    background: transparent;
    border: none;
    color: var(--text-secondary);
    font-family: var(--font-primary);
    font-size: 0.9rem;
    font-weight: 600;
    padding: 0.6rem 1.25rem;
    border-radius: var(--border-radius-sm);
    cursor: pointer;
    display: flex;
    align-items: center;
    gap: 0.5rem;
    transition: all var(--transition-fast);
}
.tab-btn i {
    font-size: 0.95rem;
}
.tab-btn:hover {
    color: var(--text-primary);
    background: rgba(255, 255, 255, 0.05);
}
[data-theme="light"] .tab-btn:hover {
    background: rgba(0, 0, 0, 0.03);
}
.tab-btn.active {
    color: #ffffff;
    background: var(--primary-gradient);
    box-shadow: 0 4px 12px rgba(14, 165, 233, 0.25);
}
/* Header Action Buttons */
.header-actions {
    display: flex;
    align-items: center;
    gap: 1rem;
}
.action-btn-circle {
    background: var(--bg-surface-solid);
    border: 1px solid var(--border-color);
    color: var(--text-secondary);
    width: 40px;
    height: 40px;
    border-radius: 50%;
    cursor: pointer;
    display: flex;
    align-items: center;
    justify-content: center;
    transition: all var(--transition-fast);
}
.action-btn-circle:hover {
    color: var(--text-primary);
    border-color: var(--border-color-hover);
    transform: translateY(-2px);
    box-shadow: var(--shadow-sm);
}
/* Theme Toggle Config */
#theme-toggle .light-icon {
    display: none;
}
#theme-toggle .dark-icon {
    display: block;
}
[data-theme="light"] #theme-toggle .light-icon {
    display: block;
}
[data-theme="light"] #theme-toggle .dark-icon {
    display: none;
}
/* ==========================================================================
   LAYOUT: MAIN CONTAINER & TABS
   ========================================================================== */
.main-container {
    flex: 1;
    padding: 2rem 2.5rem;
    max-width: 1600px;
    width: 100%;
    margin: 0 auto;
    display: flex;
    flex-direction: column;
    gap: 2rem;
}
.tab-content {
    display: none;
    animation: fadeIn var(--transition-normal);
}
.tab-content.active {
    display: block;
}
@keyframes fadeIn {
    from {
        opacity: 0;
        transform: translateY(10px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}
/* Glass panel base */
.glass-panel {
    background: var(--bg-surface);
    backdrop-filter: var(--glass-blur);
    -webkit-backdrop-filter: var(--glass-blur);
    border: 1px solid var(--border-color);
    border-radius: var(--border-radius-md);
    box-shadow: var(--shadow-md);
    transition: background var(--transition-normal), border var(--transition-normal), box-shadow var(--transition-normal);
}
.glass-panel:hover {
    border-color: var(--border-color-hover);
}
/* Panel Headers */
.panel-header {
    display: flex;
    align-items: center;
    gap: 0.75rem;
    padding: 1.5rem;
    border-bottom: 1px solid var(--border-color);
}
.panel-header h2 {
    font-family: var(--font-secondary);
    font-size: 1.25rem;
    font-weight: 700;
    color: var(--text-primary);
}
.header-icon {
    font-size: 1.25rem;
    color: var(--primary);
}
/* ==========================================================================
   TAB 1: ESTIMATOR LAYOUT
   ========================================================================== */
.estimator-layout {
    display: grid;
    grid-template-columns: 380px 1fr;
    gap: 1.5rem;
    align-items: start;
    margin-bottom: 1.5rem;
}
/* Forms & Inputs */
.input-pane {
    position: sticky;
    top: 90px;
}
.form-container {
    padding: 1.5rem;
    display: flex;
    flex-direction: column;
    gap: 1.25rem;
}
.form-group {
    display: flex;
    flex-direction: column;
    gap: 0.5rem;
}
.form-row {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 1rem;
}
.form-group label {
    font-size: 0.85rem;
    font-weight: 600;
    color: var(--text-secondary);
    letter-spacing: 0.2px;
}
.input-wrapper {
    position: relative;
    display: flex;
    align-items: center;
}
.input-icon {
    position: absolute;
    left: 1rem;
    color: var(--text-muted);
    font-size: 0.95rem;
    pointer-events: none;
    transition: color var(--transition-fast);
}
.input-wrapper input,
.input-wrapper select {
    width: 100%;
    background: rgba(0, 0, 0, 0.2);
    border: 1px solid var(--border-color);
    padding: 0.75rem 1rem 0.75rem 2.5rem;
    border-radius: var(--border-radius-sm);
    color: var(--text-primary);
    font-family: var(--font-primary);
    font-size: 0.95rem;
    outline: none;
    transition: all var(--transition-fast);
}
[data-theme="light"] .input-wrapper input,
[data-theme="light"] .input-wrapper select {
    background: rgba(255, 255, 255, 0.8);
}
.input-wrapper input:focus,
.input-wrapper select:focus {
    border-color: var(--primary);
    box-shadow: 0 0 0 3px rgba(14, 165, 233, 0.15);
    background: rgba(0, 0, 0, 0.25);
}
[data-theme="light"] .input-wrapper input:focus,
[data-theme="light"] .input-wrapper select:focus {
    background: #ffffff;
}
.input-wrapper input:focus + .input-icon,
.input-wrapper select:focus + .input-icon {
    color: var(--primary);
}
.input-help {
    font-size: 0.75rem;
    color: var(--text-muted);
}
/* Button systems */
.btn {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    gap: 0.5rem;
    font-family: var(--font-primary);
    font-size: 0.95rem;
    font-weight: 600;
    padding: 0.75rem 1.5rem;
    border-radius: var(--border-radius-sm);
    border: none;
    cursor: pointer;
    transition: all var(--transition-fast);
}
.btn-primary {
    background: var(--primary-gradient);
    color: #ffffff;
    box-shadow: 0 4px 12px rgba(14, 165, 233, 0.25);
}
.btn-primary:hover {
    background: var(--primary-gradient-hover);
    transform: translateY(-1px);
    box-shadow: 0 6px 16px rgba(14, 165, 233, 0.35);
}
.btn-secondary {
    background: var(--bg-surface-solid);
    border: 1px solid var(--border-color);
    color: var(--text-primary);
}
.btn-secondary:hover {
    border-color: var(--border-color-hover);
    background: rgba(255, 255, 255, 0.05);
}
[data-theme="light"] .btn-secondary:hover {
    background: rgba(0, 0, 0, 0.02);
}
.btn-danger-text {
    background: transparent;
    color: var(--accent-rose);
    border: 1px solid transparent;
}
.btn-danger-text:hover {
    background: rgba(244, 63, 94, 0.1);
    border-color: rgba(244, 63, 94, 0.2);
}
.btn-sm {
    padding: 0.4rem 0.8rem;
    font-size: 0.8rem;
    border-radius: 6px;
}
.form-actions {
    display: flex;
    flex-direction: column;
    gap: 0.75rem;
    margin-top: 0.5rem;
}
/* Results section styling */
.results-pane {
    display: flex;
    flex-direction: column;
    gap: 1.5rem;
}
/* KPIs Grid */
.kpi-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 1rem;
}
.kpi-card {
    padding: 1.5rem;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
    position: relative;
    overflow: hidden;
    height: 140px;
}
.kpi-card::before {
    content: '';
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 3px;
    background: var(--border-color-hover);
}
.kpi-card.highlight-cost::before {
    background: var(--primary-gradient);
}
.kpi-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    color: var(--text-secondary);
    font-size: 0.85rem;
    font-weight: 600;
    letter-spacing: 0.3px;
}
.kpi-icon {
    font-size: 1.1rem;
    color: var(--text-muted);
}
.highlight-cost .kpi-icon {
    color: var(--primary);
}
.kpi-value {
    font-family: var(--font-secondary);
    font-size: 1.85rem;
    font-weight: 800;
    color: var(--text-primary);
    margin: 0.5rem 0;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}
.highlight-cost .kpi-value {
    background: var(--primary-gradient);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    filter: drop-shadow(0 2px 4px rgba(14, 165, 233, 0.15));
}
.kpi-subtext {
    font-size: 0.75rem;
    color: var(--text-muted);
}
/* Charts Grid */
.charts-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 1.5rem;
}
.chart-container {
    padding: 1.5rem;
    display: flex;
    flex-direction: column;
    gap: 1rem;
    min-height: 340px;
}
.chart-container h3 {
    font-family: var(--font-secondary);
    font-size: 1.05rem;
    font-weight: 700;
    display: flex;
    align-items: center;
    gap: 0.5rem;
    color: var(--text-secondary);
}
.chart-container h3 i {
    color: var(--primary);
}
.chart-wrapper {
    position: relative;
    flex: 1;
    width: 100%;
    height: 100%;
    min-height: 240px;
    display: flex;
    align-items: center;
    justify-content: center;
}
/* ==========================================================================
   HISTORY TABLE SECTION
   ========================================================================== */
.history-section {
    padding: 1.5rem;
}
.section-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 1.25rem;
}
.header-left {
    display: flex;
    align-items: center;
    gap: 0.75rem;
}
.header-left i {
    font-size: 1.25rem;
    color: var(--primary);
}
.header-left h2 {
    font-family: var(--font-secondary);
    font-size: 1.25rem;
    font-weight: 700;
}
.table-container {
    overflow-x: auto;
    width: 100%;
}
.history-table {
    width: 100%;
    border-collapse: collapse;
    text-align: left;
    font-size: 0.9rem;
}
.history-table th {
    background: rgba(0, 0, 0, 0.2);
    padding: 1rem;
    font-weight: 600;
    color: var(--text-secondary);
    border-bottom: 1px solid var(--border-color);
}
[data-theme="light"] .history-table th {
    background: rgba(0, 0, 0, 0.03);
}
.history-table td {
    padding: 1rem;
    border-bottom: 1px solid var(--border-color);
    color: var(--text-primary);
    vertical-align: middle;
}
.history-table tbody tr {
    transition: background-color var(--transition-fast);
}
.history-table tbody tr:hover {
    background: rgba(255, 255, 255, 0.02);
}
[data-theme="light"] .history-table tbody tr:hover {
    background: rgba(0, 0, 0, 0.01);
}
/* Badges for Obra / Padrão */
.badge {
    display: inline-flex;
    align-items: center;
    padding: 0.25rem 0.6rem;
    border-radius: 20px;
    font-size: 0.75rem;
    font-weight: 700;
    text-transform: uppercase;
}
.badge-res { background: rgba(14, 165, 233, 0.15); color: #38bdf8; }
.badge-com { background: rgba(168, 85, 247, 0.15); color: #c084fc; }
.badge-ind { background: rgba(245, 158, 11, 0.15); color: #fbbf24; }
.badge-low { background: rgba(100, 116, 139, 0.15); color: #94a3b8; }
.badge-med { background: rgba(16, 185, 129, 0.15); color: #34d399; }
.badge-high { background: rgba(244, 63, 94, 0.15); color: #fb7185; }
.empty-state {
    text-align: center;
    color: var(--text-muted);
}
.empty-state td {
    padding: 3rem 1rem !important;
}
.cell-actions {
    display: flex;
    gap: 0.5rem;
}
.btn-action {
    background: var(--bg-surface-solid);
    border: 1px solid var(--border-color);
    color: var(--text-secondary);
    width: 32px;
    height: 32px;
    border-radius: 6px;
    display: flex;
    align-items: center;
    justify-content: center;
    cursor: pointer;
    transition: all var(--transition-fast);
    font-size: 0.85rem;
}
.btn-action:hover {
    color: var(--text-primary);
    border-color: var(--border-color-hover);
}
.btn-action.load:hover {
    color: var(--primary);
    box-shadow: 0 0 10px rgba(14, 165, 233, 0.2);
}
.btn-action.report:hover {
    color: var(--accent-emerald);
    box-shadow: 0 0 10px rgba(16, 185, 129, 0.2);
}
.btn-action.delete:hover {
    color: var(--accent-rose);
    box-shadow: 0 0 10px rgba(244, 63, 94, 0.2);
}
.materials-summary-cell {
    font-size: 0.8rem;
    color: var(--text-secondary);
    line-height: 1.4;
}
/* ==========================================================================
   TAB 2: PARAMETERS EDITOR
   ========================================================================== */
.parameters-container {
    padding: 2rem;
    display: flex;
    flex-direction: column;
    gap: 1.5rem;
}
.subtitle {
    font-size: 0.9rem;
    color: var(--text-secondary);
    margin-top: 0.25rem;
}
.tabs-subnav {
    display: flex;
    border-bottom: 1px solid var(--border-color);
    gap: 1.5rem;
}
.sub-tab-btn {
    background: transparent;
    border: none;
    border-bottom: 2px solid transparent;
    color: var(--text-secondary);
    padding: 0.75rem 0.5rem;
    cursor: pointer;
    font-family: var(--font-primary);
    font-weight: 600;
    font-size: 0.95rem;
    transition: all var(--transition-fast);
}
.sub-tab-btn:hover {
    color: var(--text-primary);
}
.sub-tab-btn.active {
    color: var(--primary);
    border-bottom-color: var(--primary);
}
.sub-tab-content {
    display: none;
    animation: fadeIn var(--transition-normal);
}
.sub-tab-content.active {
    display: block;
}
.table-info {
    display: flex;
    align-items: center;
    gap: 0.5rem;
    background: rgba(14, 165, 233, 0.08);
    border: 1px solid rgba(14, 165, 233, 0.15);
    padding: 0.75rem 1rem;
    border-radius: var(--border-radius-sm);
    color: var(--primary);
    font-size: 0.85rem;
    margin-bottom: 1.25rem;
}
.params-table {
    width: 100%;
    border-collapse: collapse;
    text-align: left;
}
.params-table th {
    padding: 1rem;
    font-weight: 600;
    color: var(--text-secondary);
    border-bottom: 2px solid var(--border-color);
}
.params-table td {
    padding: 1rem;
    border-bottom: 1px solid var(--border-color);
    vertical-align: middle;
}
.param-input {
    width: 120px;
    background: rgba(0, 0, 0, 0.15);
    border: 1px solid var(--border-color);
    border-radius: 6px;
    padding: 0.5rem 0.75rem;
    color: var(--text-primary);
    font-family: var(--font-primary);
    font-size: 0.9rem;
    font-weight: 600;
    outline: none;
    transition: all var(--transition-fast);
}
[data-theme="light"] .param-input {
    background: rgba(255, 255, 255, 0.9);
}
.param-input:focus {
    border-color: var(--primary);
    box-shadow: 0 0 0 3px rgba(14, 165, 233, 0.15);
}
/* ==========================================================================
   TAB 3: DOCUMENTATION & ONE PAGE LAYOUT
   ========================================================================== */
.docs-layout {
    display: grid;
    grid-template-columns: 280px 1fr;
    gap: 2rem;
    align-items: start;
}
.docs-sidebar {
    padding: 1.5rem;
    position: sticky;
    top: 90px;
}
.docs-sidebar h3 {
    font-family: var(--font-secondary);
    font-size: 1.1rem;
    font-weight: 700;
    color: var(--text-secondary);
    margin-bottom: 1rem;
    padding-bottom: 0.5rem;
    border-bottom: 1px solid var(--border-color);
}
.docs-sidebar ul {
    list-style: none;
    display: flex;
    flex-direction: column;
    gap: 0.25rem;
}
.docs-sidebar ul a {
    display: flex;
    align-items: center;
    gap: 0.75rem;
    color: var(--text-secondary);
    text-decoration: none;
    padding: 0.6rem 0.8rem;
    border-radius: var(--border-radius-sm);
    font-size: 0.9rem;
    font-weight: 600;
    transition: all var(--transition-fast);
}
.docs-sidebar ul a:hover {
    color: var(--text-primary);
    background: rgba(255, 255, 255, 0.03);
}
[data-theme="light"] .docs-sidebar ul a:hover {
    background: rgba(0, 0, 0, 0.02);
}
.docs-sidebar ul a.active {
    color: var(--primary);
    background: rgba(14, 165, 233, 0.08);
}
/* Docs Main Content Container */
.docs-content {
    padding: 2.5rem;
    min-height: 600px;
}
.doc-section {
    display: none;
    animation: fadeIn var(--transition-normal);
}
.doc-section.active {
    display: block;
}
.doc-section h2 {
    font-family: var(--font-secondary);
    font-size: 1.85rem;
    font-weight: 800;
    margin-bottom: 1.25rem;
    letter-spacing: -0.5px;
}
.section-lead {
    font-size: 1.1rem;
    color: var(--text-secondary);
    margin-bottom: 2rem;
    line-height: 1.6;
}
.doc-section p {
    line-height: 1.7;
    margin-bottom: 1.25rem;
    color: var(--text-secondary);
}
.doc-section h3 {
    font-family: var(--font-secondary);
    font-size: 1.35rem;
    font-weight: 700;
    margin: 2rem 0 1rem;
}
.doc-badge {
    display: inline-block;
    font-size: 0.7rem;
    font-weight: 700;
    letter-spacing: 1px;
    background: rgba(14, 165, 233, 0.15);
    color: var(--primary);
    padding: 0.3rem 0.75rem;
    border-radius: 4px;
    margin-bottom: 0.75rem;
}
/* One Page Card Grid */
.one-page-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 1.5rem;
    margin-top: 1.5rem;
}
.op-card {
    background: rgba(0, 0, 0, 0.15);
    border: 1px solid var(--border-color);
    border-radius: var(--border-radius-md);
    padding: 1.5rem;
}
[data-theme="light"] .op-card {
    background: rgba(255, 255, 255, 0.4);
}
.op-card h4 {
    font-family: var(--font-secondary);
    font-size: 1.05rem;
    font-weight: 700;
    margin-bottom: 0.75rem;
    display: flex;
    align-items: center;
    gap: 0.5rem;
}
.op-card p, .op-card ul, .op-card ol {
    font-size: 0.88rem;
    margin-bottom: 0;
    line-height: 1.6;
}
.op-card ul, .op-card ol {
    padding-left: 1.25rem;
}
.op-card li {
    margin-bottom: 0.5rem;
    color: var(--text-secondary);
}
/* Flow Chart Styling */
.flow-chart-container {
    display: flex;
    align-items: center;
    gap: 1rem;
    margin: 1.5rem 0;
}
.flow-step {
    flex: 1;
    background: rgba(0, 0, 0, 0.1);
    border: 1px solid var(--border-color);
    border-radius: var(--border-radius-sm);
    padding: 1.25rem;
    text-align: center;
    position: relative;
}
[data-theme="light"] .flow-step {
    background: rgba(255, 255, 255, 0.5);
}
.step-num {
    position: absolute;
    top: -12px;
    left: 50%;
    transform: translateX(-50%);
    background: var(--primary-gradient);
    color: #ffffff;
    width: 24px;
    height: 24px;
    border-radius: 50%;
    font-weight: 700;
    font-size: 0.8rem;
    display: flex;
    align-items: center;
    justify-content: center;
}
.flow-step h5 {
    font-family: var(--font-secondary);
    font-size: 0.95rem;
    font-weight: 700;
    margin-bottom: 0.5rem;
}
.flow-step p {
    font-size: 0.8rem;
    margin-bottom: 0;
    line-height: 1.4;
}
.flow-arrow {
    color: var(--text-muted);
    font-size: 1.25rem;
}
/* Stack Table */
.stack-table {
    width: 100%;
    border-collapse: collapse;
    margin: 1.5rem 0;
    font-size: 0.9rem;
}
.stack-table th {
    background: rgba(0, 0, 0, 0.15);
    padding: 0.75rem 1rem;
    font-weight: 700;
    border-bottom: 2px solid var(--border-color);
}
[data-theme="light"] .stack-table th {
    background: rgba(0, 0, 0, 0.02);
}
.stack-table td {
    padding: 0.75rem 1rem;
    border-bottom: 1px solid var(--border-color);
    color: var(--text-secondary);
}
/* Formulas boxes */
.formula-box {
    background: rgba(0, 0, 0, 0.15);
    border-left: 4px solid var(--primary);
    border-radius: 0 var(--border-radius-sm) var(--border-radius-sm) 0;
    padding: 1.25rem;
    margin: 1rem 0 1.5rem;
}
[data-theme="light"] .formula-box {
    background: rgba(0, 0, 0, 0.02);
}
.formula-box h4 {
    font-family: var(--font-secondary);
    font-size: 0.95rem;
    font-weight: 700;
    margin-bottom: 0.5rem;
    color: var(--text-primary);
}
.formula-box code {
    font-family: monospace;
    font-size: 0.9rem;
    color: var(--primary-hover);
    display: block;
}
/* Roadmap items */
.roadmap-list {
    display: flex;
    flex-direction: column;
    gap: 1.25rem;
}
.roadmap-item {
    background: rgba(0, 0, 0, 0.1);
    border: 1px solid var(--border-color);
    border-radius: var(--border-radius-sm);
    padding: 1.5rem;
    position: relative;
}
[data-theme="light"] .roadmap-item {
    background: rgba(255, 255, 255, 0.5);
}
.badge-road {
    background: rgba(99, 102, 241, 0.15);
    color: #818cf8;
    position: absolute;
    top: 1.5rem;
    right: 1.5rem;
}
.roadmap-item h4 {
    font-family: var(--font-secondary);
    font-size: 1.1rem;
    font-weight: 700;
    margin-bottom: 0.5rem;
}
.roadmap-item p {
    font-size: 0.88rem;
    margin-bottom: 0;
}
/* ==========================================================================
   TOAST NOTIFICATION COMPONENT
   ========================================================================== */
.toast-container {
    position: fixed;
    bottom: 2rem;
    right: 2rem;
    display: flex;
    flex-direction: column;
    gap: 0.75rem;
    z-index: 1000;
    max-width: 380px;
    width: calc(100% - 4rem);
}
.toast {
    background: var(--bg-surface-solid);
    border: 1px solid var(--border-color);
    border-left: 4px solid var(--primary);
    border-radius: var(--border-radius-sm);
    padding: 1rem 1.25rem;
    display: flex;
    align-items: center;
    gap: 0.75rem;
    box-shadow: var(--shadow-lg);
    animation: slideIn var(--transition-fast) forwards;
    cursor: pointer;
    transition: transform var(--transition-fast);
}
.toast:hover {
    transform: translateY(-2px);
}
.toast.success { border-left-color: var(--accent-emerald); }
.toast.info { border-left-color: var(--primary); }
.toast.error { border-left-color: var(--accent-rose); }
.toast.warning { border-left-color: var(--accent-amber); }
.toast-icon {
    font-size: 1.1rem;
    flex-shrink: 0;
}
.toast.success .toast-icon { color: var(--accent-emerald); }
.toast.info .toast-icon { color: var(--primary); }
.toast.error .toast-icon { color: var(--accent-rose); }
.toast.warning .toast-icon { color: var(--accent-amber); }
.toast-message {
    font-size: 0.88rem;
    font-weight: 500;
    color: var(--text-primary);
    line-height: 1.4;
}
@keyframes slideIn {
    from {
        opacity: 0;
        transform: translateY(20px) scale(0.95);
    }
    to {
        opacity: 1;
        transform: translateY(0) scale(1);
    }
}
@keyframes fadeOut {
    to {
        opacity: 0;
        transform: translateY(-10px) scale(0.95);
    }
}
/* ==========================================================================
   RESPONSIVE LAYOUTS (MEDIA QUERIES)
   ========================================================================== */
@media (max-width: 1200px) {
    .estimator-layout {
        grid-template-columns: 1fr;
    }

    .input-pane {
        position: static;
    }
}
@media (max-width: 900px) {
    .main-header {
        flex-direction: column;
        gap: 1.25rem;
        padding: 1.25rem 1.5rem;
    }

    .nav-tabs {
        width: 100%;
        overflow-x: auto;
        justify-content: flex-start;
    }

    .tab-btn {
        flex: 1;
        justify-content: center;
        white-space: nowrap;
    }

    .main-container {
        padding: 1.5rem 1.5rem;
    }

    .charts-grid {
        grid-template-columns: 1fr;
    }

    .docs-layout {
        grid-template-columns: 1fr;
    }

    .docs-sidebar {
        position: static;
        width: 100%;
    }

    .docs-sidebar ul {
        flex-direction: row;
        overflow-x: auto;
        padding-bottom: 0.5rem;
    }

    .docs-sidebar ul a {
        white-space: nowrap;
    }

    .one-page-grid {
        grid-template-columns: 1fr;
    }

    .flow-chart-container {
        flex-direction: column;
        gap: 1.5rem;
    }

    .flow-arrow {
        transform: rotate(90deg);
        margin: -0.5rem 0;
    }
}
@media (max-width: 600px) {
    .kpi-grid {
        grid-template-columns: 1fr;
    }

    .form-row {
        grid-template-columns: 1fr;
    }

    .params-table th, .params-table td {
        padding: 0.75rem 0.5rem;
    }

    .param-input {
        width: 90px;
    }
}


SyntaxError: invalid character '²' (U+00B2) (3940890725.py, line 190)